In [ ]:
import pandas as pd
fpp = "../data/olist_prepared/freq_prod_weekly_sale_SP_2017.parquet"
df = pd.read_parquet(fpp)

In [ ]:
A = df.values

In [ ]:
import numpy as np
import numpy.linalg as la

$\mathbf{M}$ is the bipartite representation of the adjacency matrix per [Indrajit Dhillon's paper](https://www.cs.utexas.edu/~inderjit/public_papers/kdd_bipartite.pdf)

In [ ]:
n = A.shape[1]
m = A.shape[0]
a = np.zeros((m,m))
b = np.zeros((n,n))
M = np.block([[a, A],[A.T, b]]) 

In [ ]:
D_diag = np.sum(M, axis=1)

In [ ]:
D_diag.shape

In [ ]:
L = D_diag - M

### Computation of the Normalized Laplacian

In [ ]:
D1 = np.sum(A, axis=1)
D2 = np.sum(A.T, axis=1)
# Calculate D1^(-1/2)
D1 = np.squeeze(np.asarray(D1))
D2 = np.squeeze(np.asarray(D2))
D1_inv_sqrt = np.diag(1.0 / np.sqrt(D1))
D2_inv_sqrt = np.diag(1.0 / np.sqrt(D2))


### Implementation per [Indrajit Dhillon's paper](https://www.cs.utexas.edu/~inderjit/public_papers/kdd_bipartite.pdf) , Algorithm Multipartition

In [ ]:
An = D1_inv_sqrt@A@D2_inv_sqrt

In [ ]:
U, s, Vt = np.linalg.svd(An)

In [ ]:
idx_sv = s.argsort()
sorted_singular_values = s[idx_sv]

In [ ]:

# 2. Set a threshold
threshold = 0.01

# 3. Create a boolean mask
mask = sorted_singular_values < threshold
true_count = np.count_nonzero(mask)
sorted_singular_values[:true_count]=0

In [ ]:
EVAL_LIMIT = 10
eig_val_idx = [(i+1) for i in range(2*EVAL_LIMIT)]
spec_gap = {"eig_val_index": eig_val_idx, "eig_vals": sorted_singular_values[:2*EVAL_LIMIT]}
df_sg = pd.DataFrame.from_dict(spec_gap, orient="columns")
df_sg["gap"] = df_sg["eig_vals"].diff()

In [ ]:
sel_gap = df_sg.gap ==  df_sg.gap.max()
df_sg[sel_gap]

In [ ]:
import plotly.express as px
fig = px.scatter(df_sg, x="eig_val_index", y="eig_vals")
fig.update_layout(title_text="Spectral Gap Identification for Olist Shoppers in Sau Paulo in 2017",
                  xaxis_range=[8, 19], width=700, height=500)

fig.show()


In [ ]:
from math import ceil, log
k = 16 # from spectral gap analysis
l = ceil(log(k,2))

In [ ]:
Ul = D1_inv_sqrt @ U
Vl = D2_inv_sqrt @ Vt

In [ ]:
Ul = Ul[:, 1:(l+1)]
Vl = Vl[:, 1:(l+1)]

In [ ]:
Z = np.vstack((Ul, Vl))

In [ ]:
Z.shape

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=k, random_state=0, n_init="auto").fit(Z)

In [ ]:
df_Z = pd.DataFrame(Z)
df_Z.columns = ["emb-" + str(i+1) for i in range(l)]
df_Z["cluster"] = kmeans.labels_

In [ ]:
df_Z.cluster.value_counts()

In [ ]:
df_prod = df_Z.loc[Ul.shape[0]:, :]

In [ ]:
df_week = df_Z.loc[:Ul.shape[0], :]

In [ ]:
df_week.cluster.unique()

In [ ]:
df_prod.cluster.unique()

In [ ]:
df_prod.cluster.value_counts()

In [ ]:
df_prod.shape

In [ ]:
cluster_sel = df_prod.cluster == 8

In [ ]:
products = df.columns.tolist()

In [ ]:
len(products)

In [ ]:

cluster_prods = [item for item, select in zip(products, cluster_sel) if select]

In [ ]:
weeks = df.index.tolist()

In [ ]:
df_week.cluster.value_counts()

In [ ]:
cluster_sel = df_week.cluster == 3
weeks_in_cluster = [item for item, select in zip(weeks, cluster_sel) if select]

In [ ]:
weeks_in_cluster